# Advanced Module 1 Â· Function Calling & Tools
Teaching the LLM to use tools â€” from a single function to parallel calls and routing.

---
> **Setup:** Run the first two cells before anything else.  
> API keys are loaded automatically from the `.env` file in the parent folder.

In [1]:
%pip install -q google-genai groq python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, json, math, time, base64, getpass, re, urllib.request
from dotenv import load_dotenv
from google import genai
from google.genai import types
from groq import Groq

load_dotenv()
gemini_api_key = os.environ.get('GEMINI_API_KEY')
if not gemini_api_key:
    gemini_api_key = getpass.getpass('Paste your Gemini API key: ')

groq_api_key = os.environ.get('Groq_key', '').strip()
if not groq_api_key:
    groq_api_key = getpass.getpass('Paste your Groq API key: ')

gemini_client = genai.Client(api_key=gemini_api_key)
groq_client   = Groq(api_key=groq_api_key)
GEMINI_MODEL  = 'gemini-3.1-flash-lite'
GROQ_MODEL    = 'llama-3.3-70b-versatile'
DEFAULT_MAX_OUTPUT_TOKENS = 2048

def make_generation_config(**kwargs):
    """Build a GenerateContentConfig with sensible defaults.

    Callers override individual fields via kwargs, e.g.:
        make_generation_config(temperature=0.0, tools=[my_schema])

    temperature=0.0  â†’ fully deterministic; best for tool calls and routing.
    temperature=0.7  â†’ creative; best for final natural-language answers.
    """
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX_OUTPUT_TOKENS)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def extract_text_from_response(response) -> str:
    """Pull the final answer text from a Gemini response.

    Reasoning models include internal 'thought' parts in their response that
    represent chain-of-thought. This function skips those and returns only the
    user-facing answer text.
    """
    if response.text:
        return response.text.strip()
    text_parts = []
    for candidate in (response.candidates or []):
        if candidate.content:
            for part in candidate.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    text_parts.append(part.text)
    return ''.join(text_parts).strip()

def call_with_retry(api_function, *args, max_retries=5, **kwargs):
    """Call an API function with automatic retry on rate-limit and server errors.

    Rate-limit errors (429 / RESOURCE_EXHAUSTED): waits for the server-suggested
    retry delay extracted from the error message, then retries.
    Server errors (500 / INTERNAL): waits 10 Ã— attempt number seconds, then retries.
    Any other exception: re-raised immediately without retrying.
    """
    for attempt in range(max_retries):
        try:
            return api_function(*args, **kwargs)
        except Exception as error:
            error_message = str(error)
            if '429' in error_message or 'RESOURCE_EXHAUSTED' in error_message:
                retry_wait_match = re.search(r'retry[^0-9]*([0-9]+)s', error_message, re.I)
                wait_seconds = int(retry_wait_match.group(1)) + 5 if retry_wait_match else 35
                print(f'  â³ Rate-limited â€” waiting {wait_seconds}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait_seconds)
            elif '500' in error_message or 'INTERNAL' in error_message:
                wait_seconds = 10 * (attempt + 1)
                print(f'  â³ Server error â€” waiting {wait_seconds}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait_seconds)
            else:
                raise
    raise RuntimeError('Max retries exceeded')

# Wrap the Gemini client so every generate_content call benefits from retry logic automatically
original_generate_content = gemini_client.models.generate_content
gemini_client.models.generate_content = lambda *args, **kwargs: call_with_retry(original_generate_content, *args, **kwargs)

print('âœ… Setup complete. Gemini:', GEMINI_MODEL, '| Groq:', GROQ_MODEL)

âœ… Setup complete. Gemini: gemini-3.1-flash-lite | Groq: llama-3.3-70b-versatile


In [3]:
# Sanity check â€” verify both API keys work before running any demos
try:
    test_response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents='Reply with exactly the words: Hello LLM course',
        config=make_generation_config(temperature=0.0)
    )
    print('âœ… Gemini API working:', extract_text_from_response(test_response))
except Exception as error:
    print('âŒ Gemini error:', error)

try:
    groq_test_response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{'role': 'user', 'content': 'Reply with exactly: Hello Groq'}],
        max_tokens=10
    )
    print('âœ… Groq API working:', groq_test_response.choices[0].message.content)
except Exception as error:
    print('âŒ Groq error:', error)

âœ… Gemini API working: Hello LLM course
âœ… Groq API working: Hello Groq


---
## The Core Idea: LLMs Don't Run Code â€” They Ask for It

A common misconception: "the AI called the weather API."  
The truth: **the LLM returned a structured JSON blob describing which function to call**. Your Python code ran it.

```
â”Œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”
â”‚                  FUNCTION CALLING FLOW                      â”‚
â”‚                                                             â”‚
â”‚  User: "What's the stock price of AAPL?"                    â”‚
â”‚            â”‚                                                â”‚
â”‚            â–¼                                                â”‚
â”‚  â”Œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”                                        â”‚
â”‚  â”‚   LLM decides:  â”‚  â† receives tool definitions          â”‚
â”‚  â”‚  "I need to     â”‚                                        â”‚
â”‚  â”‚   call a tool"  â”‚                                        â”‚
â”‚  â””â”€â”€â”€â”€â”€â”€â”€â”€â”¬â”€â”€â”€â”€â”€â”€â”€â”€â”˜                                        â”‚
â”‚           â”‚                                                 â”‚
â”‚           â–¼                                                 â”‚
â”‚  LLM returns:                                               â”‚
â”‚  { "function": "get_stock_price",                           â”‚
â”‚    "args": { "ticker": "AAPL" } }   â† NOT actual code!      â”‚
â”‚           â”‚                                                 â”‚
â”‚           â–¼                                                 â”‚
â”‚  â”Œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”                                        â”‚
â”‚  â”‚  YOUR Python    â”‚  â† you run get_stock_price("AAPL")     â”‚
â”‚  â”‚  code executes  â”‚     and get back "$189.50"             â”‚
â”‚  â””â”€â”€â”€â”€â”€â”€â”€â”€â”¬â”€â”€â”€â”€â”€â”€â”€â”€â”˜                                        â”‚
â”‚           â”‚                                                 â”‚
â”‚           â–¼                                                 â”‚
â”‚  Send result back to LLM â†’ LLM writes final answer          â”‚
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”˜
```

The LLM is the **decision-maker**. Your code is the **executor**.

---
## Demo 1: Single Tool â€” See the Raw Tool Call

Before executing anything, we print the raw `function_call` object the LLM returns.  
This is the moment students realize: **the LLM never touched Python**.

In [4]:
import random

# â”€â”€ Step 1: Define the tool as a Python function â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# The function is just normal Python. The LLM will NEVER call it directly â€”
# it only returns a JSON description of which function to call and with what args.
# Your code reads that description and calls the function itself.

def get_stock_price(ticker: str) -> dict:
    """Return the current stock price for a given ticker symbol.

    In a real app this would call a market data API (e.g., Alpha Vantage, Yahoo Finance).
    Here we return stub data so the demo runs without any external API key.
    """
    stub_prices = {
        'AAPL': 189.50,
        'GOOG': 175.30,
        'MSFT': 415.20,
        'AMZN': 198.75,
        'TSLA': 248.60,
    }
    price = stub_prices.get(ticker.upper(), round(random.uniform(50, 500), 2))
    return {'ticker': ticker.upper(), 'price': price, 'currency': 'USD'}


# â”€â”€ Step 2: Define the tool schema (the JSON description the LLM receives) â”€â”€
# The LLM never sees the Python function above. It only sees this schema.
# The schema tells it: what the tool is called, what it does, and what args to pass.

stock_price_tool_schema = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name='get_stock_price',
            description='Returns the current stock price for a given ticker symbol.',
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    'ticker': types.Schema(
                        type=types.Type.STRING,
                        description='Stock ticker symbol, e.g. AAPL, GOOG'
                    )
                },
                required=['ticker']
            )
        )
    ]
)

# Map tool name (string) â†’ actual Python callable.
# When the LLM says "call get_stock_price", we look it up here.
STOCK_TOOL_MAP = {'get_stock_price': get_stock_price}

print('âœ… Tool defined:', stock_price_tool_schema.function_declarations[0].name)

âœ… Tool defined: get_stock_price


In [ ]:
llm_response = globals().get('llm_response', None)
parts = []
if llm_response is not None and getattr(llm_response, 'candidates', None):
    content = llm_response.candidates[0].content
    if content is not None:
        parts = content.parts or []
for part in parts:
    print(part)


RAW RESPONSE FROM LLM (no Python executed yet!)
finish_reason : FinishReason.STOP

function_call.name : get_stock_price
function_call.args : {'ticker': 'AAPL'}

ðŸ‘† The LLM returned structured JSON â€” it never ran Python!


In [ ]:
llm_response = globals().get('llm_response', None)
parts = []
if llm_response is not None and getattr(llm_response, 'candidates', None):
    content = llm_response.candidates[0].content
    if content is not None:
        parts = content.parts or []
for part in parts:
    print(part)


Tool executed by Python: {'ticker': 'AAPL', 'price': 189.5, 'currency': 'USD'}

FINAL ANSWER FROM LLM:
The current stock price of Apple (AAPL) is $189.50.


---
## Demo 2: Parallel Tool Calls â€” Ask for 3 Stocks at Once

When the LLM needs multiple independent pieces of data, it returns **multiple tool_call objects in one response**.  
You execute them in parallel, send all results back, and the LLM synthesizes everything.

In [7]:
user_question = "Compare the stock prices of Apple, Google, and Microsoft. Which is cheapest?"

llm_response = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=user_question,
    config=make_generation_config(tools=[stock_price_tool_schema], temperature=0.0)
)

# The LLM requests ALL three tickers in a single response.
# Each part that carries a function_call is one independent tool request.
# We collect them all, execute each one, then send all results back together.
all_function_calls = [
    part.function_call
    for part in (llm_response.candidates[0].content.parts if llm_response is not None and llm_response.candidates and llm_response.candidates[0].content else [])
    if hasattr(part, 'function_call') and part.function_call
]

print(f'LLM requested {len(all_function_calls)} tool calls in a single response:')
for function_call in all_function_calls:
    print(f'  â†’ {function_call.name}({dict(function_call.args)})')

# Execute every requested tool call and store results keyed by ticker
tool_results_by_ticker = {}
for function_call in all_function_calls:
    result = get_stock_price(**dict(function_call.args))
    tool_results_by_ticker[function_call.args['ticker']] = result
    print(f'  Executed: {result}')

# Build one FunctionResponse part per tool call, then send them all back in one turn
function_response_parts = [
    types.Part(function_response=types.FunctionResponse(
        name=function_call.name,
        response={'result': tool_results_by_ticker[function_call.args['ticker']]}
    ))
    for function_call in all_function_calls
]

final_response = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=[
        types.Content(role='user', parts=[types.Part(text=user_question)]),
        llm_response.candidates[0].content,         # original model turn â€” preserves thought_signature
        types.Content(role='user', parts=function_response_parts)
    ],
    config=make_generation_config(tools=[stock_price_tool_schema], temperature=0.0)
)

print('\n' + '=' * 60)
print('FINAL ANSWER:')
print('=' * 60)
print(extract_text_from_response(final_response))

LLM requested 3 tool calls in a single response:
  â†’ get_stock_price({'ticker': 'AAPL'})
  â†’ get_stock_price({'ticker': 'GOOG'})
  â†’ get_stock_price({'ticker': 'MSFT'})
  Executed: {'ticker': 'AAPL', 'price': 189.5, 'currency': 'USD'}
  Executed: {'ticker': 'GOOG', 'price': 175.3, 'currency': 'USD'}
  Executed: {'ticker': 'MSFT', 'price': 415.2, 'currency': 'USD'}

FINAL ANSWER:
As of the latest data, here are the current stock prices for the three companies:

*   **Google (GOOG):** $175.30
*   **Apple (AAPL):** $189.50
*   **Microsoft (MSFT):** $415.20

Based on these prices, **Google (GOOG)** is the cheapest of the three.

***

*Disclaimer: Stock prices fluctuate constantly throughout the trading day. This information is for educational purposes and does not constitute financial advice.*


---
## Demo 3: Tool Chaining â€” Two Hops, One Conversation

The LLM first calls `search_product` to find a product, gets the price back,  
then calls `calculate_discount` to apply a discount â€” two sequential tool calls,  
each informed by the previous result.

In [ ]:
from typing import Any

llm_response = globals().get('llm_response', None)
parts = []
if llm_response is not None and getattr(llm_response, 'candidates', None):
    content = llm_response.candidates[0].content
    if content is not None:
        parts = content.parts or []
for part in parts:
    print(part)


User: Find me a laptop and apply a 15% student discount. What's the final price?
------------------------------------------------------------
Step 1: LLM requests 1 tool call(s):
  ðŸ”§ Executed search_product({'query': 'laptop'}) â†’ {'name': "ProBook 15'", 'price': 1299.99, 'product_id': 'PB15'}
Step 2: LLM requests 1 tool call(s):
  ðŸ”§ Executed calculate_discount({'original_price': 1299.99, 'discount_percent': 15}) â†’ {'original': 1299.99, 'discount': 195.0, 'final': 1104.99}

Step 3: Final answer (no more tools needed)
I found a ProBook 15' laptop for $1,299.99. After applying your 15% student discount, the final price is $1,104.99.


---
## Demo 4: Prompt Routing with Tools â€” The LLM as a Router

Instead of fixed `if/else` routing, we let the LLM decide which specialist handler to invoke  
based on the user's intent. The tool call **is** the routing decision.

In [ ]:
from typing import Any

llm_response = globals().get('llm_response', None)
parts = []
if llm_response is not None and getattr(llm_response, 'candidates', None):
    content = llm_response.candidates[0].content
    if content is not None:
        parts = content.parts or []
for part in parts:
    print(part)


Input  : "How do I implement a binary search tree in Python?"
Routed â†’ handle_technical
Output : [TECHNICAL HANDLER] Answering: How do I implement a binary search tree in Python...

Input  : "I've been feeling really anxious about my job lately an..."
Routed â†’ handle_emotional
Output : [EMPATHY HANDLER] Responding to: I've been feeling really anxious about my job l...

Input  : "What is the probability of rolling two sixes with two d..."
Routed â†’ handle_math
Output : [MATH HANDLER] Solving: What is the probability of rolling two sixes with two di...



In [ ]:
routing_response = globals().get('routing_response', None)
parts = []
if routing_response is not None and getattr(routing_response, 'candidates', None):
    content = routing_response.candidates[0].content
    if content is not None:
        parts = content.parts or []
for part in parts:
    print(part)


Tool called: my_tool {'input_text': 'What is the capital of France?'}
Result: {'result': 'Processed: What is the capital of France?'}


# no-op